# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets and their fields
# Each RecordSet and Field are referenced by their @id

record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"RecordSet #{i+1} : @id = {rs.id}")
    print(f"  Name      : {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id} ({field.name})")
    print('-'*50)
if not record_sets:
    print("No explicit record sets registered in the Croissant metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, record set IDs are referenced from the metadata

if record_sets:
    record_set_ids = [rs.id for rs in record_sets]
else:
    # If no explicit record sets found in metadata, try to list available ones by resource or fallback to a default
    # (Croissant datasets sometimes put all fields in one default recordset)
    record_set_ids = []

# We'll load data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f'--- Loading records for RecordSet: {record_set_id} ---')
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Fields / columns in {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print('  No records found for this RecordSet.')
if not dataframes:
    print('No dataframes loaded. Check metadata or use the default record set (@id).')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Find a numeric field (by @id) and perform analysis using the first available RecordSet
import numpy as np
import warnings
warnings.filterwarnings("ignore")

if dataframes and record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]

    # Try to automatically select a plausible numeric field
    # We'll select the first numeric field as per the schema
    numeric_field_id = None
    group_field = None

    field_types = {}
    for rs in metadata.record_sets:
        if rs.id == main_record_set_id:
            for field in rs.fields:
                field_types[field.id] = getattr(field, 'data_type', None)
                if field_types[field.id] in ['Number', 'Integer', 'Float', 'schema:Number', 'schema:Float', 'schema:Integer'] and not numeric_field_id:
                    numeric_field_id = field.id
                if field_types[field.id] == 'Text' and not group_field:
                    group_field = field.id
    
    if not numeric_field_id:
        # Try to auto-detect one from dataframe
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field_id = col
                break
    if not group_field:
        for col in df.columns:
            if df[col].dtype == object:
                group_field = col
                break

    print(f"Selected numeric_field_id = {numeric_field_id}")
    print(f"Selected group_field    = {group_field}")

    # Filter records where the numeric field is greater than some threshold
    threshold = df[numeric_field_id].quantile(0.75) if numeric_field_id else None
    if numeric_field_id and threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field and take mean
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print('No data loaded to perform EDA steps.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and record_set_ids and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field or data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. We:
- Loaded Croissant metadata and summarized the dataset.
- Listed available record sets and their fields, referencing all entities by their `@id`.
- Extracted tabular data and performed basic filtering, normalization, and grouping using pandas.
- Visualized data distributions and categorical breakdowns.

**Next steps:** Further analysis can include more advanced modeling or external enrichment as appropriate for your research question.